# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [45]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [46]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
 # 对每条 y（反转后的序列）做：'[0]' + y[:-1]'
#前面加一个 0，当作 起始符 / BOS（这里用 0 表示）。
#后面是 真实输出 y 去掉最后一个元素，即每一步的输入是「上一步的正确输出」。
#这样在训练时：第 t步用 dec_x[t] 预测 y[t]（标准 seq2seq 错位对齐）。
print(get_batch(2, 10))

(['NAIDOUZTXX', 'BEIADBLCNJ'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[14,  1,  9,  4, 15, 21, 26, 20, 24, 24],
       [ 2,  5,  9,  1,  4,  2, 12,  3, 14, 10]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 24, 24, 20, 26, 21, 15,  4,  9,  1],
       [ 0, 10, 14,  3, 12,  2,  4,  1,  9,  5]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[24, 24, 20, 26, 21, 15,  4,  9,  1, 14],
       [10, 14,  3, 12,  2,  4,  1,  9,  5,  2]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [47]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        # 编码器（与 sequence_reversal-exercise 一致）
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)

        # 解码器：逐步 Luong 式双线性 attention + SimpleRNNCell（与参考中 embed→cell→dense 对齐，多 enc 上下文）
        dec_emb = self.embed_layer(dec_ids)
        dec_len = tf.shape(dec_ids)[1]
        attn_ta = tf.TensorArray(
            dtype=tf.float32, size=dec_len,
            element_shape=tf.TensorShape([None, self.hidden]))
        decoder_state = enc_state

        for t in tf.range(dec_len):
            dec_input_t = dec_emb[:, t, :]
            query = tf.expand_dims(decoder_state, 1)
            score = tf.matmul(self.dense_attn(query), enc_out, transpose_b=True)
            attn_weights = tf.nn.softmax(score, axis=-1)
            context = tf.matmul(attn_weights, enc_out)
            context = context[:, 0, :]
            decoder_in = tf.concat([dec_input_t, context], axis=-1)
            out_t, new_states = self.decoder_cell(decoder_in, [decoder_state])
            decoder_state = new_states[0]
            attn_ta = attn_ta.write(t, out_t)

        dec_outputs = tf.transpose(attn_ta.stack(), [1, 0, 2])
        logits = self.dense(dec_outputs)
        return logits

    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]

    def get_next_token(self, x, state, enc_out):
        '''shape(x) = [b_sz,]；encode 传入的 state 为 list 时，须用 state[1](enc_state) 与 call() 初值 enc_state 一致'''
        if isinstance(state, (list, tuple)):
            decoder_state = state[1]
        else:
            decoder_state = state
        inp_emb = self.embed_layer(x)
        query = tf.expand_dims(decoder_state, 1)
        score = tf.matmul(self.dense_attn(query), enc_out, transpose_b=True)
        attn_weights = tf.nn.softmax(score, axis=-1)
        context = tf.matmul(attn_weights, enc_out)
        context = context[:, 0, :]
        decoder_in = tf.concat([inp_emb, context], axis=-1)
        h, new_states = self.decoder_cell.call(decoder_in, [decoder_state])
        logits = self.dense(h)
        out = tf.argmax(logits, axis=-1)
        return out, new_states[0]

# Loss函数以及训练逻辑

In [48]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [49]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.312437
step 500 : loss 0.9713799
step 1000 : loss 0.4572137
step 1500 : loss 0.057814706


<tf.Tensor: shape=(), dtype=float32, numpy=0.031730693>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [50]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, False, True, True, True, True, True, True, True, True, True]
[('LSXUVTLWESSOLMOTWUCV', 'VCUWTOMLOSSEWLTVUXSL'), ('ESXPQYUDPCLCRMXHLBHB', 'QHBLHXMRCLCPDUYQPXSE'), ('PAKBZRZPSLKDDZAYYUTG', 'GTUYYAZDDKLSPZRZBKAP'), ('ETKTCIJEYTIZKRENOJSC', 'CSJONERKZITYEJICTKTE'), ('UFLHQHWTPJWIPPFQOREV', 'VEROQFPPIWJPTWHQHLFU'), ('NLQMDHNUJLQXJEAKCHVU', 'UVHCKAEJXQLJUNHDMQLN'), ('DKWFHEEUAMFFBUBFTOSP', 'PSOTFBUBFFMAUEEHFWKD'), ('ONQXJKBJZHUXZTDBIGCE', 'ECGIBDTZXUHZJBKJXQNO'), ('XTJJGNAKXVGZSGXHJJNU', 'UNJJHXGSZGVXKANGJJTX'), ('RLKDBQTKHTLZCOZHMOOX', 'XOOMHZOCZLTHKTQBDKLR'), ('XJIMEHQFEQFXJBCKZSTD', 'DTSZKCBJXFQEFQHEMIJX'), ('TILEPYLTFSRWIQUJJTCG', 'GCTJJUQIWRSFTLYPELIT'), ('CRZIFHNIRGNRGLSWNDDD', 'DDDNWSLGRNGRINHFIZRC'), ('NODKQROPISUYDHNJOPQM', 'MQPOJNHDYUSIPORQKDON'), ('VKLBSSTFEKQFFIQBGVTB', 'BTVGBQIFFQKEFTSSBLKV'), ('UZUZICPOJZIEOHNBFFXD', 'DXFFBNHOEIZJOPCIZUZU'), ('HO